In [1]:
pip install --upgrade openai python-dotenv pandas tqdm tenacity

  Using cached openai-2.15.0-py3-none-any.whl (1.1 MB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
     |████████████████████████████████| 12.8 MB 14.6 MB/s            
  Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl (28 kB)
  Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
     |████████████████████████████████| 113 kB 112.8 MB/s            
  Using cached distro-1.9.0-py3-none-any.whl (20 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
     |████████████████████████████████| 366 kB 56.2 MB/s            
  Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
     |████████████████████████████████| 19.5 MB 72.8 MB/s            
     |████████████████████████████████| 348 kB 103.8 MB/s            
  Using cached idna-3.11-py3-none-any.whl (71 kB)
     |████████████████████████████████| 152 kB 84.6 MB/s            
  Using cached httpcore-1.0.9-py3-no

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

from openai import OpenAI


In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://api.mistral.ai/v1",
    api_key=os.getenv("MISTRAL_API_KEY"),
)

# Mistral Large 3 (v25.12)
MODEL = "mistral-large-2512"


In [5]:
def build_user_prompt(title: str, body: str) -> str:
    return f"""
      Your response must be under 120 words. Give a reply to the OP for the following post:

TITLE: {title}

BODY:
{body}
"""


In [6]:
@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(min=1, max=20),
    retry=retry_if_exception_type(Exception),
)
def generate_reply(title: str, body: str) -> str:
    full_prompt = build_user_prompt(title, body)

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": full_prompt}],
        temperature=0.2,
    )

    return (resp.choices[0].message.content or "").strip()



In [7]:
TEST_TITLE = "is this sa?"
TEST_BODY = (
    """
    
        okay i feel ridiculous even typing this but i keep spiraling and i need someone to reality-check me.

so i was hanging out with a friend last night at his apartment. we were watching a movie on the couch and i was kinda tired so i leaned back and he ended up sitting closer than usual. at one point he asked “is this okay?” and i said “yeah it’s fine” because i genuinely didn’t mind. he put his arm around the back of the couch (not like around me, just behind me) and my brain immediately went into panic mode like “wait did i accidentally agree to something” even though nothing else happened.

we didn’t kiss, he didn’t touch me in a weird way, he didn’t say anything sexual, and when the movie ended i left like normal. he even texted after like “home safe?” and that was it.

but now i’m stuck replaying it and thinking: did i consent to something i didn’t mean to? was it weird that he asked “is this okay” like that? i’ve had anxiety all day and i feel sick even though logically it sounds like nothing.

am i overthinking this? i just want to make sure i’m not missing some red flag or something.

    """
)

reply = generate_reply(TEST_TITLE, TEST_BODY)
print(reply)


It sounds like you’re overthinking this—your friend checked in, you felt comfortable, and nothing inappropriate happened. His question was just him being considerate, not a red flag. Anxiety can twist small moments into big worries, but logically, this was just a normal interaction. If you’re still uneasy, trust your gut and talk to him about it casually. You’re not missing anything—just your brain playing tricks on you. 💙


In [8]:
TEST_TITLE = "I don’t know if what happened was assault"
TEST_BODY = (
    "I was with someone I trust and things escalated. I didn’t say no, but I also didn’t really want it. "
    "Now I feel confused and guilty calling it assault. Part of me thinks I’m overreacting, but I can’t stop thinking about it."
)

reply = generate_reply(TEST_TITLE, TEST_BODY)
print(reply)


It’s completely valid to feel confused—your feelings matter more than labels right now. If you didn’t fully want it, that’s worth acknowledging, even if you didn’t say no. Trust and consent go beyond just words; your discomfort is important. You’re not overreacting—this is a heavy thing to process. Consider talking to someone you trust or a professional who can help you sort through it. Be gentle with yourself. 💛


In [9]:
from pathlib import Path

# Input / output paths
IN_PATH = Path("../../data/Ambivalent_w_comments.csv")
OUT_PATH = Path("../../data/Ambivalent_w_comments_with_llm_reply_mistral.csv")

# Load
df = pd.read_csv(IN_PATH)

# Ensure title/body are strings (avoid NaN issues)
df["title"] = df["title"].fillna("").astype(str)
df["body"]  = df["body"].fillna("").astype(str)

# Generate replies
llm_replies = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    try:
        r = generate_reply(row["title"], row["body"])
    except Exception as e:
        r = f"ERROR: {type(e).__name__}: {e}"
    llm_replies.append(r)

df["LLM_Reply"] = llm_replies

# Keep only the requested columns (in this exact order)
out_cols = [
    "post_id",
    "subreddit",
    "title",
    "author",
    "score",
    "num_comments_listed",
    "op_replied",
    "op_reply_count",
    "created_utc",
    "permalink",
    "body",
    "LLM_Reply",
]

df_out = df[out_cols]
df_out.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH.resolve()}")

  0%|          | 0/1011 [00:00<?, ?it/s]

100%|██████████| 1011/1011 [49:53<00:00,  2.96s/it] 

Saved: /home/ec2-user/ambivalent/data/Ambivalent_w_comments_with_llm_reply_mistral.csv
